# Interchange

> Conversion to and from the Hypergraph Interchange Format (HIF)

[HIF](https://github.com/pszufe/HIF-standard) is a JSON standard for exchanging higher-order networks between libraries e.g. [XGI](https://github.com/xgi-org/xgi) and [HyperNetX](https://github.com/pnnl/HyperNetX) (Coll et al., 2025, *Network Science* 13; [arXiv:2507.11520](https://arxiv.org/abs/2507.11520)). The JSON schema in `parag/data/hif_schema.json` is from the HIF-standard repository (MIT License).

In [ ]:
#| default_exp io

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
# | export
import json
import logging
from itertools import combinations

import pandas as pd


def _to_records(df: pd.DataFrame) -> list:
    """JSON-safe records (numpy scalars to Python types, NaN to None).

    Args:
        df (pd.DataFrame): table

    Returns:
        list: records
    """
    return json.loads(df.to_json(orient="records"))


def _to_ids(ds: pd.Series) -> list:
    """HIF-compatible ids (integers or strings).

    Args:
        ds (pd.Series): ids

    Returns:
        list: ids
    """
    if ds.isna().any():
        raise ValueError(f"missing ids in '{ds.name}'")
    if pd.api.types.is_integer_dtype(ds):
        return [int(i) for i in ds]
    return [str(i) for i in ds]


def _to_attrs(df: pd.DataFrame, cols: list) -> list:
    """`attrs` dicts of the HIF records.

    Args:
        df (pd.DataFrame): table
        cols (list): columns to keep as attributes

    Returns:
        list: one dict per row (empty if there are no attributes)
    """
    if len(cols) == 0:
        return [{} for _ in range(len(df))]
    return _to_records(df.loc[:, cols])

In [ ]:
# | export
def to_hif(
    nodes: pd.DataFrame,
    edges: pd.DataFrame,
    col_node_id: str,
    col_source: str,
    col_target: str,
    col_subset_id: str = None,
    mode: str = "subsets",
    directed: bool = False,
    metadata: dict = None,
) -> dict:
    """Convert node and edge tables to the Hypergraph Interchange Format (HIF).

    Args:
        nodes (pd.DataFrame): nodes
        edges (pd.DataFrame): pairwise edges
        col_node_id (str): column with node ids
        col_source (str): column with sources
        col_target (str): column with targets
        col_subset_id (str, optional): column with subset ids; required for `mode='subsets'`. Defaults to None.
        mode (str, optional): 'subsets': hyperedges are the subsets of nodes, pairwise edges are kept in the metadata; 'pairwise': each pairwise edge is a hyperedge of 2 nodes. Defaults to 'subsets'.
        directed (bool, optional): mark sources as 'tail' and targets as 'head' (`mode='pairwise'` only). Defaults to False.
        metadata (dict, optional): additional metadata. Defaults to None.

    Returns:
        dict: HIF
    """
    if mode not in ["subsets", "pairwise"]:
        raise ValueError(f"mode should be 'subsets' or 'pairwise', got '{mode}'")
    if directed and mode != "pairwise":
        raise ValueError("directed=True requires mode='pairwise'")
    if mode == "subsets" and col_subset_id is None:
        raise ValueError("col_subset_id is required for mode='subsets'")
    cols_missing = [c for c in [col_node_id, col_subset_id] if c is not None and c not in nodes] + [
        c for c in [col_source, col_target] if c not in edges
    ]
    if len(cols_missing) > 0:
        raise KeyError(f"columns not found: {cols_missing}")
    if nodes[col_node_id].duplicated().any():
        raise ValueError(f"duplicated node ids in '{col_node_id}'")

    ## nodes
    cols_node_attrs = [c for c in nodes if c != col_node_id and (mode == "pairwise" or c != col_subset_id)]
    hif_nodes = [
        {"node": n, **({"attrs": a} if a else {})}
        for n, a in zip(_to_ids(nodes[col_node_id]), _to_attrs(nodes, cols_node_attrs))
    ]
    hif = {
        "network-type": "directed" if directed else "undirected",
        "metadata": {**(metadata if metadata is not None else {})},
        "nodes": hif_nodes,
    }
    cols = dict(
        node_columns=nodes.columns.tolist(),
        edge_columns=edges.columns.tolist(),
        mode=mode,
        directed=directed,
        col_node_id=col_node_id,
        col_source=col_source,
        col_target=col_target,
        col_subset_id=col_subset_id,
    )
    if mode == "subsets":
        ## hyperedges: subsets
        df1 = nodes.loc[:, [col_node_id, col_subset_id]]
        if df1[col_subset_id].isna().any():
            raise ValueError(f"missing subset ids in '{col_subset_id}'")
        hif["edges"] = [{"edge": e} for e in dict.fromkeys(_to_ids(df1[col_subset_id]))]
        hif["incidences"] = [
            {"edge": e, "node": n} for n, e in zip(_to_ids(df1[col_node_id]), _to_ids(df1[col_subset_id]))
        ]
        ## pairwise edges, for a lossless round trip
        hif["metadata"]["parag"] = {**cols, "edges": _to_records(edges)}
    else:
        ## hyperedges: pairwise edges
        cols_edge_attrs = [c for c in edges if c not in [col_source, col_target]]
        sources, targets = _to_ids(edges[col_source]), _to_ids(edges[col_target])
        hif["edges"] = [
            {"edge": e, **({"attrs": a} if a else {})} for e, a in enumerate(_to_attrs(edges, cols_edge_attrs))
        ]
        hif["incidences"] = []
        for e, (s, t) in enumerate(zip(sources, targets)):
            if directed:
                hif["incidences"] += [
                    {"edge": e, "node": s, "direction": "tail"},
                    {"edge": e, "node": t, "direction": "head"},
                ]
            else:
                # g: a self-loop is one incidence
                hif["incidences"] += [{"edge": e, "node": n} for n in dict.fromkeys([s, t])]
        hif["metadata"]["parag"] = cols
    return hif

In [ ]:
# | export
def _from_parag_hif(hif: dict) -> dict:
    """Restore the tables of a HIF written by `to_hif`.

    Args:
        hif (dict): HIF with `metadata['parag']`

    Returns:
        dict: nodes, edges and their column names
    """
    meta = hif["metadata"]["parag"]
    df1 = pd.DataFrame(
        [{meta["col_node_id"]: d["node"], **d.get("attrs", {})} for d in hif.get("nodes", [])]
    )
    if meta["mode"] == "subsets":
        node2subset = {d["node"]: d["edge"] for d in hif["incidences"]}
        df1 = df1.assign(**{meta["col_subset_id"]: lambda df: df[meta["col_node_id"]].map(node2subset)})
        df2 = pd.DataFrame(meta["edges"])
    else:
        edge2incidences = {}
        for d in hif["incidences"]:
            edge2incidences.setdefault(d["edge"], []).append(d)
        records = []
        for d in hif["edges"]:
            incidences = edge2incidences[d["edge"]]
            if meta["directed"]:
                direction2node = {i["direction"]: i["node"] for i in incidences}
                s, t = direction2node["tail"], direction2node["head"]
            else:
                # g: a self-loop has one incidence
                s, t = incidences[0]["node"], incidences[-1]["node"]
            records.append({meta["col_source"]: s, meta["col_target"]: t, **d.get("attrs", {})})
        df2 = pd.DataFrame(records)
    return dict(
        nodes=df1.loc[:, meta["node_columns"]],
        edges=df2.loc[:, meta["edge_columns"]],
        **{k: meta[k] for k in ["col_node_id", "col_source", "col_target", "col_subset_id"]},
    )


def from_hif(
    hif: dict,
    subset_rule: str = None,
) -> dict:
    """Convert a Hypergraph Interchange Format (HIF) dict to node and edge tables, e.g. for `to_net`.

    Hyperedges are the subsets of the nodes. Pairwise edges are obtained by clique expansion of the hyperedges (tails x heads for directed networks). HIFs written by `to_hif` are restored as they were.

    Args:
        hif (dict): HIF
        subset_rule (str, optional): subset of the nodes that belong to multiple hyperedges: 'largest' (hyperedge with the most nodes) or 'first' (first incidence). Raises an error if None. Defaults to None.

    Returns:
        dict: nodes, edges and their column names, e.g. `to_net(**from_hif(hif))`
    """
    if "parag" in hif.get("metadata", {}):
        return _from_parag_hif(hif)
    if subset_rule not in [None, "largest", "first"]:
        raise ValueError(f"subset_rule should be None, 'largest' or 'first', got '{subset_rule}'")
    directed = hif.get("network-type", "undirected") == "directed"
    df0 = pd.DataFrame(hif["incidences"], columns=["edge", "node", "direction"])
    if directed and df0["direction"].isna().any():
        logging.warning(
            f"{df0['direction'].isna().sum()} incidence/s without direction in a directed network; excluded from the pairwise edges."
        )

    ## subsets
    df1 = df0.drop_duplicates(subset=["edge", "node"]).assign(
        **{"edge size": lambda df: df.groupby("edge")["node"].transform("size")}
    )
    node_count = df1["node"].duplicated(keep=False).groupby(df1["node"]).any().sum()
    if node_count > 0:
        if subset_rule is None:
            raise ValueError(
                f"{node_count} node/s belong to multiple hyperedges; set subset_rule='largest' or 'first' to select one subset per node."
            )
        logging.warning(f"{node_count} node/s belong to multiple hyperedges; selected the '{subset_rule}' one.")
        if subset_rule == "largest":
            df1 = df1.sort_values("edge size", ascending=False, kind="stable")
    node2subset = df1.drop_duplicates(subset=["node"], keep="first").set_index("node")["edge"].to_dict()

    ## nodes
    df2 = pd.DataFrame(
        [{"node id": d["node"], **d.get("attrs", {})} for d in hif.get("nodes", [])]
    )
    if "node id" not in df2:
        df2 = pd.DataFrame(columns=["node id"])
    nodes_missing = [n for n in dict.fromkeys(df0["node"]) if n not in set(df2["node id"])]
    df2 = pd.concat([df2, pd.DataFrame({"node id": nodes_missing})], axis=0, ignore_index=True).assign(
        subset=lambda df: df["node id"].map(node2subset)
    )

    ## edges: clique expansion
    records = []
    for e, df in df0.groupby("edge", sort=False):
        if directed:
            pairs = [
                (s, t)
                for s in df.loc[df["direction"] == "tail", "node"]
                for t in df.loc[df["direction"] == "head", "node"]
            ]
        else:
            members = list(dict.fromkeys(df["node"]))
            pairs = list(combinations(members, 2)) if len(members) > 1 else [(members[0], members[0])]
        records += [{"source": s, "target": t, "hyperedge id": e} for s, t in pairs]
    df3 = pd.DataFrame(records, columns=["source", "target", "hyperedge id"])
    edge2attrs = {d["edge"]: d["attrs"] for d in hif.get("edges", []) if "attrs" in d}
    if len(edge2attrs) > 0:
        df3 = df3.join(pd.DataFrame([edge2attrs.get(e, {}) for e in df3["hyperedge id"]], index=df3.index))
    return dict(
        nodes=df2,
        edges=df3,
        col_node_id="node id",
        col_source="source",
        col_target="target",
        col_subset_id="subset",
    )

In [ ]:
# | export
def validate_hif(
    hif: dict,
    errors: str = "raise",
) -> bool:
    """Validate a HIF dict against the HIF JSON schema (requires `jsonschema`).

    Args:
        hif (dict): HIF
        errors (str, optional): 'raise' the validation error, else log it as a warning. Defaults to 'raise'.

    Returns:
        bool: whether the HIF is valid
    """
    import jsonschema
    from roux.lib.io import read_dict

    from parag.utils import get_src_path

    schema = read_dict(f"{get_src_path()}/data/hif_schema.json")
    try:
        jsonschema.validate(instance=hif, schema=schema)
    except jsonschema.ValidationError as e:
        if errors == "raise":
            raise
        logging.warning(f"invalid HIF: {e.message}")
        return False
    return True


def write_hif(
    hif: dict,
    output_path: str,
) -> str:
    """Write a HIF dict to a JSON file, after validating it.

    Args:
        hif (dict): HIF
        output_path (str): output path (.json)

    Returns:
        str: output path
    """
    from roux.lib.io import to_dict

    validate_hif(hif)
    return to_dict(hif, output_path)


def read_hif(
    input_path: str,
) -> dict:
    """Read a HIF JSON file, after validating it.

    Args:
        input_path (str): input path (.json)

    Returns:
        dict: HIF
    """
    from roux.lib.io import read_dict

    hif = read_dict(input_path)
    validate_hif(hif)
    return hif

## Tests

In [ ]:
#| hide
import json
import tempfile
import networkx as nx
import xgi

Round trip of the network communities (undirected), in both modes, and reading by XGI.

In [ ]:
G = nx.karate_club_graph()
nodes = (
    pd.Series({i: list(t) for i, t in enumerate(list(nx.community.girvan_newman(G))[3])})
    .explode()
    .to_frame("node id")
    .reset_index()
    .rename(columns={"index": "community id"})
    .astype({"node id": int})
    .loc[:, ["node id", "community id"]]
)
edges = pd.DataFrame(G.edges, columns=["source", "target"])
for mode in ["subsets", "pairwise"]:
    hif = to_hif(nodes, edges, "node id", "source", "target", "community id", mode=mode)
    assert validate_hif(hif)
    d = from_hif(json.loads(json.dumps(hif)))
    pd.testing.assert_frame_equal(d["nodes"], nodes, check_dtype=False)
    pd.testing.assert_frame_equal(d["edges"], edges, check_dtype=False)
    H = xgi.convert.from_hif_dict(hif)
    assert H.num_nodes == len(nodes)
    assert H.num_edges == (nodes["community id"].nunique() if mode == "subsets" else len(edges))

Directed pairwise edges (drug → side effect), and writing/reading a HIF file.

In [ ]:
edges = pd.read_csv("examples/inputs/ChSe-Decagon_monopharmacy.csv")
nodes = pd.concat(
    [
        edges.loc[:, ["# STITCH"]].drop_duplicates().rename(columns={"# STITCH": "node id"}).assign(subset="drug"),
        edges.loc[:, ["Individual Side Effect", "Side Effect Name"]]
        .drop_duplicates(subset=["Individual Side Effect"])
        .rename(columns={"Individual Side Effect": "node id"})
        .assign(subset="side effect"),
    ],
    axis=0,
    ignore_index=True,
)
hif = to_hif(nodes, edges, "node id", "# STITCH", "Individual Side Effect", "subset", mode="pairwise", directed=True)
with tempfile.TemporaryDirectory() as dir_path:
    d = from_hif(read_hif(write_hif(hif, f"{dir_path}/drugs.json")))
pd.testing.assert_frame_equal(d["nodes"], nodes, check_dtype=False)
pd.testing.assert_frame_equal(d["edges"], edges, check_dtype=False)
assert type(xgi.convert.from_hif_dict(hif)).__name__ == "DiHypergraph"

A HIF written by another library (XGI), with nodes in multiple hyperedges.

In [ ]:
hif = dict(xgi.convert.to_hif_dict(xgi.Hypergraph([[1, 2, 3], [3, 4], [4, 5, 6, 7]])))
try:
    from_hif(hif)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass
d = from_hif(hif, subset_rule="largest")
assert d["nodes"]["subset"].tolist() == [0, 0, 0, 2, 2, 2, 2]
assert len(d["edges"]) == 3 + 1 + 6  # g: clique expansion
assert from_hif(hif, subset_rule="first")["nodes"]["subset"].tolist() == [0, 0, 0, 1, 2, 2, 2]

Invalid HIF.

In [ ]:
assert not validate_hif({"nodes": []}, errors="warn")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()